In [17]:
import os
import pandas as pd
import subprocess as sp
import polars as pl

In [18]:
out_dir = "../../results/04_single_cell_access_atac/16_split_fragment"
os.makedirs(out_dir, exist_ok=True)

In [19]:
df_meta = pd.read_csv("../../results/04_single_cell_access_atac/14_create_seurat/snATAC_metadata.csv", index_col=0)

In [20]:
df_meta.head()

,orig.ident,nCount_ATAC,nFeature_ATAC,Sample,TSSEnrichment,ReadsInTSS,ReadsInPromoter,PromoterRatio,PassQC,NucleosomeRatio,...,predicted.id,prediction.score.Club,prediction.score.Basal,prediction.score.Ciliated,prediction.score.Tuft,prediction.score.PNEC,prediction.score.Goblet,prediction.score.Ionocyte,prediction.score.max,cell_type
mouse#GAGGCTCCAGAGTCGA,SeuratProject,33439,33439,mouse,10.229,10326,33420,0.142007,1,4.658300,...,Basal,0.115445,0.872481,0.000000,0.012074,0.0,0.0,0.0,0.872481,Club
mouse#CTGTATTTCGTATAGC,SeuratProject,35815,35815,mouse,9.869,10102,33263,0.144250,1,3.783669,...,Basal,0.158509,0.821975,0.019516,0.000000,0.0,0.0,0.0,0.821975,Basal
mouse#TATCGAGGTAACGTAA,SeuratProject,35117,35117,mouse,10.541,10338,33123,0.143647,1,3.778984,...,Basal,0.156596,0.843404,0.000000,0.000000,0.0,0.0,0.0,0.843404,Basal
mouse#CGCAGGTAGAACGTCG,SeuratProject,30979,30979,mouse,9.835,8950,29135,0.135571,1,4.227330,...,Basal,0.153923,0.840544,0.005534,0.000000,0.0,0.0,0.0,0.840544,Club
mouse#TTCTGTAGTATCCTTT,SeuratProject,30426,30426,mouse,8.935,8194,27820,0.130027,1,3.927591,...,Club,0.508905,0.491095,0.000000,0.000000,0.0,0.0,0.0,0.508905,Club


In [21]:
df_meta["barcode"] = df_meta.index
df_meta["barcode"] = df_meta["barcode"].str.split("#").str[1]
df_meta = df_meta[["barcode", "cell_type"]]

In [22]:
df_meta.head()

,barcode,cell_type
mouse#GAGGCTCCAGAGTCGA,GAGGCTCCAGAGTCGA,Club
mouse#CTGTATTTCGTATAGC,CTGTATTTCGTATAGC,Basal
mouse#TATCGAGGTAACGTAA,TATCGAGGTAACGTAA,Basal
mouse#CGCAGGTAGAACGTCG,CGCAGGTAGAACGTCG,Club
mouse#TTCTGTAGTATCCTTT,TTCTGTAGTATCCTTT,Club


In [23]:
df_fragments = pl.read_csv(
            "../../results/04_single_cell_access_atac/09_merge_fragment/merged.fragments.aggregated.tsv.gz",
            skip_rows=0,
            has_header=False,
            separator="\t",
            use_pyarrow=False,
            new_columns=["Chromosome", "Start", "End", "Barcode", "Count", "Edit"])

df_fragments = df_fragments.drop("column_7")

In [24]:
df_fragments.head()

Chromosome,Start,End,Barcode,Count,Edit
str,i64,i64,str,i64,str
"""chr1""",3050003,3050371,"""GGATAGGAGTGCTAGG""",16,"""3050117|3050122|3050123|305012…"
"""chr1""",3050004,3050131,"""TTCATCAAGAACCCGA""",14,"""3050099|3050114|3050130"""
"""chr1""",3050004,3050149,"""TTACGTAAGCACCATA""",1,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGAACCATA""",12,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGCACCATT""",1,"""3050071|3050116|3050117|305013…"


In [25]:
df_fragments = df_fragments.filter(pl.col("Chromosome").is_in(["chr1","chr2","chr3","chr4","chr5","chr6","chr7","chr8","chr9","chr10",
                                                              "chr11","chr12","chr13","chr14","chr15","chr16","chr17","chr18","chr19",
                                                              "chrX","chrY"]))

In [26]:
df_fragments.head()

Chromosome,Start,End,Barcode,Count,Edit
str,i64,i64,str,i64,str
"""chr1""",3050003,3050371,"""GGATAGGAGTGCTAGG""",16,"""3050117|3050122|3050123|305012…"
"""chr1""",3050004,3050131,"""TTCATCAAGAACCCGA""",14,"""3050099|3050114|3050130"""
"""chr1""",3050004,3050149,"""TTACGTAAGCACCATA""",1,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGAACCATA""",12,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGCACCATT""",1,"""3050071|3050116|3050117|305013…"


In [27]:
file_handles = {}
for cell_type in df_meta["cell_type"].unique():
    file_name = f"{out_dir}/{cell_type}.fragments.tsv"
    file_handles[cell_type] = open(file_name, "w")

In [28]:
for cell_type in df_meta["cell_type"].unique():
    cell_barcodes = df_meta[df_meta["cell_type"] == cell_type]["barcode"].tolist()
    _df_fragments = df_fragments.filter(pl.col("Barcode").is_in(cell_barcodes))

    for row in _df_fragments.iter_rows():
        line = "\t".join([str(x) for x in row])
        file_handles[cell_type].write(line + "\n")

    file_handles[cell_type].close()

    sp.run(["bgzip", "-f", f"{out_dir}/{cell_type}.fragments.tsv"])